# TradingAgents - FPT fresh analysis and verified report download

Notebook mới mở trực tiếp trong Google Colab. Nhánh code dùng là feature/vietnam-market-foundation và được checkout vào thư mục riêng để không ghi đè thư mục cũ.

## Cách chạy
1. Chọn Runtime > Run all để kiểm tra SSI, kiểm tra Gemini, chạy đúng một lượt phân tích FPT và tải ZIP.
2. Muốn tải lại báo cáo sau này, chỉ chạy ô cuối có tiêu đề PACKAGE + DOWNLOAD EXISTING REPORT ONLY. Ô đó không gọi Gemini hoặc SSI.
3. API keys chỉ được đọc từ Colab Secrets/runtime hoặc nhập trong ô ẩn; notebook không in hoặc ghi key vào GitHub.

Ô chạy AI là nơi duy nhất tiêu thụ hạn mức Gemini. Ô tải ZIP xác minh báo cáo có tồn tại, không rỗng, và ZIP đọc được trước khi tải.


In [ ]:
# 1) Setup: isolated checkout; does not change your old /content/TradingAgents folder.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
CODE_BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgentsFresh")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", CODE_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-B", CODE_BRANCH, "FETCH_HEAD"],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    cwd=str(REPO_DIR),
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

print(f"Repository ready: {CODE_BRANCH}")
print("Setup did not call Gemini or SSI.")


In [ ]:
# 2) Credentials and read-only SSI integration check. No order/trading endpoint is called.
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo

def read_colab_secret(name):
    current = os.environ.get(name)
    if isinstance(current, str) and current.strip():
        return current.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def get_secret(name):
    return read_colab_secret(name) or getpass(f"{name} (input hidden): ")

ssi_key = get_secret("SSI_API_KEY")
ssi_secret = get_secret("SSI_API_SECRET")
if not ssi_key or not ssi_secret:
    raise ValueError("SSI_API_KEY and SSI_API_SECRET are required; values are never printed.")
os.environ["SSI_API_KEY"] = ssi_key
os.environ["SSI_API_SECRET"] = ssi_secret
del ssi_key, ssi_secret

VN_TODAY = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
FROM_DATE = (VN_TODAY - timedelta(days=30)).isoformat()
TO_DATE = VN_TODAY.isoformat()

from tradingagents.dataflows.vietnam_market.smoke import main as ssi_smoke_main
from tradingagents.dataflows.config import set_config
from tradingagents.dataflows.router import get_vendor
from tradingagents.agents.tools import get_stock_data, get_indicators

print(f"[1/2] SSI REST read-only check: FPT, {FROM_DATE} to {TO_DATE}")
exit_code = ssi_smoke_main([
    "--symbol", "FPT", "--from-date", FROM_DATE, "--to-date", TO_DATE, "--page-size", "100"
])
if exit_code != 0:
    raise RuntimeError("SSI REST check failed. Use the sanitized status message above; do not share credentials.")

set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
price_vendor = get_vendor("core_stock_apis", "get_stock_data")
indicator_vendor = get_vendor("technical_indicators", "get_indicators")
print(f"[2/2] Router: prices={price_vendor}; indicators={indicator_vendor}")
if price_vendor.split(",")[0].strip() != "ssi" or indicator_vendor.split(",")[0].strip() != "ssi":
    raise RuntimeError("Router did not select SSI for both prices and indicators.")

price_result = get_stock_data.func("FPT", FROM_DATE, TO_DATE, TO_DATE)
if not isinstance(price_result, str) or "SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
    raise RuntimeError("Agent price tool did not return an SSI-sourced result.")
price_rows = [
    line for line in price_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not price_rows:
    raise RuntimeError("SSI price tool returned no OHLCV rows.")

indicator_result = get_indicators.func("FPT", "rsi", TO_DATE, 30, TO_DATE)
if not isinstance(indicator_result, str) or "SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
    raise RuntimeError("Agent indicator tool did not return an SSI-sourced result.")
indicator_rows = [
    line for line in indicator_result.splitlines()
    if line.strip() and not line.startswith(("#", "Date,"))
]
if not indicator_rows:
    raise RuntimeError("SSI RSI tool returned no points.")

print(f"SSI + router + agent tools passed: {len(price_rows)} OHLCV rows; {len(indicator_rows)} RSI points.")
print("No credentials were printed or written into notebook cells.")


## 3) Run one fresh AI analysis

Ô kế tiếp là ô duy nhất gọi Gemini và chạy TradingAgents. Model ưu tiên đầu tiên là gemini-3.5-flash-lite, model đã phản hồi thành công trong lượt trước. Nếu model tạm quá tải, notebook thử lại một lần rồi chuyển sang model dự phòng.

Nếu chỉ muốn tải lại báo cáo sau này, không chạy lại ô này; chỉ chạy ô cuối.


In [ ]:
# 3) Gemini preflight with bounded fallback + one end-to-end TradingAgents run.
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import time

google_key = get_secret("GOOGLE_API_KEY")
if not google_key:
    raise ValueError("GOOGLE_API_KEY is empty.")
os.environ["GOOGLE_API_KEY"] = google_key
del google_key

MODEL_CANDIDATES = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
    "gemini-3.6-flash",
]

from langchain_google_genai import ChatGoogleGenerativeAI

def safe_gemini_error(exc):
    message = str(exc)
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key:
        message = message.replace(key, "[REDACTED]")
    return message[:500]

def is_transient_error(exc):
    value = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in value for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT",
    ))

MODEL_ID = None
model_errors = []
for candidate in MODEL_CANDIDATES:
    model_ok = False
    for attempt in range(2):
        try:
            response = ChatGoogleGenerativeAI(
                model=candidate,
                google_api_key=os.environ["GOOGLE_API_KEY"],
                max_retries=0,
                max_output_tokens=24,
                timeout=45,
            ).invoke("Reply with exactly the single word READY.")
            if not getattr(response, "content", None):
                raise RuntimeError("Gemini returned an empty response.")
            MODEL_ID = candidate
            model_ok = True
            print(f"Gemini preflight passed: {MODEL_ID}")
            break
        except Exception as exc:
            model_errors.append(f"{candidate}: {type(exc).__name__}: {safe_gemini_error(exc)}")
            upper = (type(exc).__name__ + " " + str(exc)).upper()
            if any(token in upper for token in (
                "401", "403", "API_KEY_INVALID", "PERMISSION_DENIED",
                "API KEY NOT VALID", "BILLING",
            )):
                raise RuntimeError(f"Gemini authentication/project access failed: {safe_gemini_error(exc)}") from None
            if attempt == 0 and is_transient_error(exc):
                print(f"{candidate} temporarily unavailable; retrying once.")
                time.sleep(2)
                continue
            print(f"{candidate} unavailable; trying the next model.")
            break
    if model_ok:
        break

if MODEL_ID is None:
    raise RuntimeError(
        "All configured Gemini models failed. Analysis was not started.\n"
        + "\n".join(model_errors[-4:])
    )

from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.graph.trading_graph import TradingAgentsGraph

config = deepcopy(DEFAULT_CONFIG)
config["llm_provider"] = "google"
config["quick_think_provider"] = None
config["deep_think_provider"] = None
config["quick_think_llm"] = MODEL_ID
config["deep_think_llm"] = MODEL_ID
config["output_language"] = "Vietnamese"
config["max_debate_rounds"] = 1
config["max_risk_discuss_rounds"] = 1
config["max_tool_rounds"] = 4
config["max_recur_limit"] = 100
config["max_tokens"] = 2048
config["llm_max_retries"] = 1
config["data_vendors"] = dict(config.get("data_vendors", {}))
config["data_vendors"]["core_stock_apis"] = "ssi"
config["data_vendors"]["technical_indicators"] = "ssi"

ANALYSIS_DATE = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date().isoformat()
print(f"Starting one TradingAgents run: FPT / {ANALYSIS_DATE}; model={MODEL_ID}; SSI prices/indicators.")
print("This cell is the only one that runs the multi-agent analysis.")

ta = TradingAgentsGraph(selected_analysts=["market"], config=config, debug=False)
final_state, signal = ta.propagate("FPT", ANALYSIS_DATE, asset_type="stock")
report_path = Path(ta.save_reports(final_state, "FPT", html=True)).resolve()

if report_path.is_dir():
    report_dir = report_path
    report_path = report_dir / "complete_report.md"
else:
    report_dir = report_path.parent

if not report_path.is_file() or report_path.stat().st_size <= 0:
    raise RuntimeError(f"Graph completed but complete_report.md is missing/empty: {report_path}")

print("TradingAgents graph completed; complete_report.md is non-empty.")
print(f"Signal/rating: {signal}")
print(f"Report bytes: {report_path.stat().st_size:,}")
print(f"Report directory: {report_dir}")
print("The final cell packages/downloads this existing report; it will not call the model again.")


In [ ]:
# 4) PACKAGE + DOWNLOAD EXISTING REPORT ONLY.
# This cell never calls Gemini or SSI. Run only this cell to download the report again.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files as colab_files

def find_existing_fpt_report(candidate=None):
    if candidate is not None:
        p = Path(candidate).expanduser()
        if p.is_file() and (p.name.lower() == "complete_report.md" or p.suffix.lower() == ".md"):
            return p.resolve()
        if p.is_dir():
            expected = p / "complete_report.md"
            if expected.is_file():
                return expected.resolve()

    roots = [
        Path("/root/.tradingagents/logs/reports"),
        Path("/content/TradingAgentsFresh/results/reports"),
        Path("/content/results/reports"),
    ]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() == ".md":
            matches.append(root)
        elif root.is_dir():
            matches.extend(
                p for p in root.rglob("complete_report.md")
                if p.is_file() and p.parent.name.startswith("FPT_")
            )
    if not matches:
        raise FileNotFoundError(
            "No existing FPT complete_report.md found in this runtime. "
            "This cell will not rerun AI; the earlier runtime may have been reset."
        )
    return max(set(p.resolve() for p in matches), key=lambda p: p.stat().st_mtime)

candidate = globals().get("report_path")
if candidate is None:
    candidate = globals().get("report_dir")
existing_report = find_existing_fpt_report(candidate)

if existing_report.stat().st_size <= 0:
    raise RuntimeError(f"Report is empty; refusing to create ZIP: {existing_report}")

existing_dir = existing_report.parent
items = sorted(
    p for p in existing_dir.rglob("*")
    if p.is_file() and not p.is_symlink()
)
if existing_report not in items:
    items.append(existing_report)
items = sorted(set(items))
if not items:
    raise RuntimeError(f"No files found for packaging: {existing_dir}")

archive = Path("/content/TradingAgents_FPT_report.zip")
temp_archive = Path(str(archive) + ".tmp")
archive.parent.mkdir(parents=True, exist_ok=True)
temp_archive.unlink(missing_ok=True)

try:
    with ZipFile(temp_archive, "w", compression=ZIP_DEFLATED) as zf:
        for item in items:
            zf.write(item, arcname=f"{existing_dir.name}/{item.relative_to(existing_dir)}")

    with ZipFile(temp_archive, "r") as zf:
        names = zf.namelist()
        corrupt = zf.testzip()
        expected = f"{existing_dir.name}/{existing_report.relative_to(existing_dir)}"
        if not names or corrupt:
            raise RuntimeError(f"ZIP verification failed: empty={not names}; corrupt member={corrupt}")
        if expected not in names or zf.getinfo(expected).file_size <= 0:
            raise RuntimeError("ZIP verification failed: complete_report.md missing or empty")
    if temp_archive.stat().st_size <= 0:
        raise RuntimeError("ZIP archive has zero bytes.")
    temp_archive.replace(archive)
finally:
    temp_archive.unlink(missing_ok=True)

print(f"ZIP VERIFIED: {len(names)} files; {archive.stat().st_size:,} bytes.")
print(f"Report inside ZIP: {expected}; report size={existing_report.stat().st_size:,} bytes.")
for name in names:
    print(" -", name)
print("Starting browser download. This cell makes no Gemini/SSI calls.")
colab_files.download(str(archive))
